In [1]:
import pandas as pd
import duckdb

df_device = pd.DataFrame({
    "device_id": ["R05", "R16", "R34", "R36"],
    "device_name": [
        "前向散射仪",
        "云高仪",
        "跑道视程仪",
        "自动气象站"
    ],
    "site": ["05端", "16端", "34端", "36端"]
})

df_device_alarm = pd.DataFrame({
    "alarm_id": [101, 102, 103, 104, 105, 106],
    "device_id": ["R05", "R05", "R16", "R34", "R34", "R36"],
    "alarm_time": [
        "2026-07-20 08:00:00",
        "2026-07-21 09:30:00",
        "2026-07-20 10:00:00",
        "2026-07-22 07:40:00",
        "2026-07-22 08:10:00",
        "2026-07-20 11:00:00"
    ],
    "alarm_level": [
        "WARNING",
        "ERROR",
        "ERROR",
        "ERROR",
        "WARNING",
        "INFO"
    ]
})

df_maintenance_order = pd.DataFrame({
    "order_id": [201, 202, 203, 204],
    "device_id": ["R05", "R16", "R16", "R36"],
    "order_status": [
        "COMPLETED",
        "OPEN",
        "CANCELLED",
        "COMPLETED"
    ],
    "created_time": [
        "2026-07-21 10:00:00",
        "2026-07-20 11:00:00",
        "2026-07-21 14:00:00",
        "2026-07-20 12:00:00"
    ]
})

df_device_alarm["alarm_time"] = pd.to_datetime(
    df_device_alarm["alarm_time"]
)

df_maintenance_order["created_time"] = pd.to_datetime(
    df_maintenance_order["created_time"]
)

df_device

,device_id,device_name,site
0,R05,前向散射仪,05端
1,R16,云高仪,16端
2,R34,跑道视程仪,34端
3,R36,自动气象站,36端


# SQL Daily Review：查找尚无已完成维修工单的异常设备

## 题目背景

设备出现异常告警后，维修人员可能会创建维修工单。

现在需要找出满足以下两个条件的设备：

1. 至少出现过一次 `ERROR` 级别的告警；
2. 不存在状态为 `COMPLETED` 的维修工单。

## 题目要求

从设备表 `df_device` 中，找出：

```text
出现过 ERROR 告警
并且
没有任何 COMPLETED 维修工单
```

### 输出字段

| 字段 | 含义 |
|---|---|
| `device_id` | 设备编号 |
| `device_name` | 设备名称 |
| `site` | 安装位置 |

### 判断规则

判断设备是否出现过异常告警时，只考虑：

```text
alarm_level = 'ERROR'
```

判断设备是否已经完成维修时，只考虑：

```text
order_status = 'COMPLETED'
```

`OPEN` 和 `CANCELLED` 都不能视为已经完成维修。

### 最终排序

按照 `device_id` 升序排列。

## 解题要求

- 以 `df_device` 作为主表；
- 使用 `EXISTS` 判断设备是否存在 `ERROR` 告警；
- 使用 `NOT EXISTS` 判断设备是否不存在 `COMPLETED` 工单；
- 子查询必须通过 `device_id` 与外层设备表建立关联；
- 不使用 `LEFT JOIN`；
- 不使用 `GROUP BY`；
- 每台设备最终只输出一行。

In [2]:
query = '''
SELECT 
    device_id,
    device_name,
    site
FROM df_device
'''
df = duckdb.execute(query).fetchdf()
df

,device_id,device_name,site
0,R05,前向散射仪,05端
1,R16,云高仪,16端
2,R34,跑道视程仪,34端
3,R36,自动气象站,36端


In [11]:
query = '''
SELECT *
    
FROM df_device_alarm
WHERE alarm_level = 'ERROR'
'''
df = duckdb.execute(query).fetchdf()
df

,alarm_id,device_id,alarm_time,alarm_level
0,102,R05,2026-07-21 09:30:00,ERROR
1,103,R16,2026-07-20 10:00:00,ERROR
2,104,R34,2026-07-22 07:40:00,ERROR


In [10]:
query = '''
SELECT *
    
FROM df_maintenance_order
'''
df = duckdb.execute(query).fetchdf()
df

,order_id,device_id,order_status,created_time
0,201,R05,COMPLETED,2026-07-21 10:00:00
1,202,R16,OPEN,2026-07-20 11:00:00
2,203,R16,CANCELLED,2026-07-21 14:00:00
3,204,R36,COMPLETED,2026-07-20 12:00:00


In [14]:
query = """
WITH error_device AS (
    SELECT
        alarm_id,
        device_id,
        alarm_time,
        alarm_level
    FROM df_device_alarm
    WHERE alarm_level = 'ERROR'
)

SELECT
    d.device_id,
    d.device_name,
    d.site
FROM df_device AS d
WHERE EXISTS (
    SELECT 1
    FROM error_device AS ed
    WHERE ed.device_id = d.device_id
)
AND NOT EXISTS (
    SELECT 1
    FROM df_maintenance_order AS dmo
    WHERE dmo.device_id = d.device_id
      AND dmo.order_status = 'COMPLETED'
)
ORDER BY d.device_id;
"""

df = duckdb.execute(query).fetchdf()
df

,device_id,device_name,site
0,R16,云高仪,16端
1,R34,跑道视程仪,34端
